#### RAG Pipeline with LangChain

In [1]:
# import libraries
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages

In [2]:
load_dotenv()
client = OpenAI()

In [3]:
model = 'gpt-4.1-nano'
db_name = "hf_emb_db"

#### Load Vector DB created previously using HF Embedding

In [4]:
embedding = HuggingFaceEmbeddings(model="all-miniLM-L6-v2")

vector_store = Chroma(
        persist_directory=db_name,
        embedding_function=embedding
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## The Retreiver and LLM

To fetch embedding and rank them we need a `retriever` and to get response back we need a `Generative LLM`

LangChain Chroma `as_retriever()` have abstracted a lot in terms of similarity calculation and ranking. Similarly LangChain variant of the LLMs e.g. ChatOpenAI in this case have also abstracted a lot in terms of interacting with the model `openai.chat.completions.create` function.

In [5]:
retriever = vector_store.as_retriever()
llm = ChatOpenAI(temperature=0, model=model)

**NOTE:** Temperature simply controls the variety of the model's output. This has been called `creativity of the model`. What an LLM does inherently is to assign probabilities to the next likely tokens. What temperature does when set to zero is to pick the next token with the highest probability. When temperature increases say to 1, it can then select the next high rank token in terms of probability x% amount of time say 10%

### LangChain objects implement the method `invoke()`

##### invoking the retriever

In [6]:
# calling invoke on the retriever
retriever.invoke("who is Avery?")

[Document(id='13e40927-9aaf-4a13-8197-49ba27aeb576', metadata={'doc_type': 'employees', 'source': 'C:\\Users\\ADE\\OneDrive\\Documents\\deep_learning\\knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \

Calling invoke on the retriver returns the top 4 documents (chunks) with the most relevent information about the query

##### invoking the llm

In [7]:
llm.invoke("who is Avery?").content

'Could you please provide more context or specify which Avery you are referring to? There are many individuals with the name Avery, including celebrities, fictional characters, or public figures.'

**Note:** Right now the LLM is not in connection with the vector store and only returns generic answer like when call on the model's API

## Putting it all together

Give the LLM context it needs to answer user queries

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.

Context:
{context}
"""

In [9]:
# fuction will go into gradio and can be named anything other than chat
def answer_question(question, history):
    docs = retriever.invoke(question)
    context= "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
answer_question("who is Avery?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been with the company since its founding in 2015 and has played a key role in establishing Insurellm as a leading provider in the insurance technology industry. Avery is known for her innovative leadership, risk management expertise, and her active involvement in professional development, diversity initiatives, and community outreach.'

In [11]:
# with this architecture - we can even get the right answer when we spell wrongly

In [12]:
answer_question("who is Avery Lancashire?", [])

'It seems like there might be a typo in the name. Based on the information I have, you might be referring to Avery Lancaster. She is the Co-Founder and CEO of Insurellm, based in San Francisco. If you meant someone else or need more specific details, please let me know!'

### Integrating with Gradio

Chat Without History

In [14]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### Chatting with History
To help the chat understand context better we need to feed it chat history. While this is a good practice it also has its pitfall.

##### chat without history
 - who is Avery?
 - How much is her salary? --> this breaks without history.

In [15]:
def combined_question(question, history):
    """
    combines all user's messages into a single string
    """
    prior = "\n".join(
    m["content"] if isinstance(m["content"], str)
    else " ".join(b["text"] for b in m["content"] if b.get("type") == "text")
    for m in history if m["role"] == "user"
    )
    return prior + "\n" + question

In [16]:
def fetch_context(question):
    return retriever.invoke(question)

In [17]:
def answer_question_2(question, history):
    combined = combined_question(question, history)
    docs = fetch_context(combined)
    context= "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content

### chat with history fixes extended context issues.

Fetch context based on what the user has said previously and not just the recent or current query

In [18]:
gr.ChatInterface(answer_question_2).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [22]:
answer_question_2('Who is Avery and how much does lancaster earns?', [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, a leading Insurance Tech company based in San Francisco, California. As of 2023, her current salary is $225,000.'

### What happens when context changes? or A completely new question is asked?